In [1]:
import torch
from torch.utils.data import DataLoader, Subset
from transformers import T5TokenizerFast, CLIPProcessor, CLIPTokenizerFast, CLIPImageProcessorFast, T5ForConditionalGeneration


from peft import LoraConfig, get_peft_model, TaskType
from torch.amp import autocast, GradScaler
from tqdm import tqdm
import os 


In [2]:
from Modules.config import (TRAIN_IMAGE_DIR,
                            TEST_IMAGE_DIR,
                            FAISS_IMAGE_PATH,
                            TRAIN_METADATA_PATH,
                            TEST_METADATA_PATH,
                            CLIP_MODEL_NAME,
                            T5_MODEL_NAME,
                            VLM_CHECKPOINT_DIR,
                            T5_DECODER_LORA_CONFIG)

from Modules.FusionVLM import FusionVLM, create_default_FusionVLM, load_default_FusionVLM, save_FusionVLM, apply_lora_config
from Modules.retrieval_module import Retriever
from Modules.datasets import VLMDataset, VLMDataCollator
from Modules.utils import print_model_param_stats, add_dict
from Modules.metrics import evaluate_captioning, setup_nltk
from Modules.train_VLM import train_and_evaluate_model

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [4]:
CLIP_processor = CLIPImageProcessorFast.from_pretrained(CLIP_MODEL_NAME, local_files_only=True)
CLIP_tokenizer = CLIPTokenizerFast.from_pretrained(CLIP_MODEL_NAME, local_files_only=True)
T5_tokenizer = T5TokenizerFast.from_pretrained(T5_MODEL_NAME, local_files_only=True)

In [5]:
collator_T5 = VLMDataCollator(CLIP_processor, T5_tokenizer, device=DEVICE)
collator_CLIP = VLMDataCollator(CLIP_processor, CLIP_tokenizer, label_tokenizer=T5_tokenizer, max_seq_len=77, device=DEVICE)

collator = collator_T5
# collator = collator_CLIP

In [6]:
retriever = Retriever(metadata_path=TRAIN_METADATA_PATH, faiss_path=FAISS_IMAGE_PATH)

train_dataset = VLMDataset(image_dir=TRAIN_IMAGE_DIR, 
                           ref_image_dir=TRAIN_IMAGE_DIR,
                           metadata_path=TRAIN_METADATA_PATH,
                           retriever=retriever)

test_dataset = VLMDataset(image_dir=TEST_IMAGE_DIR, 
                           ref_image_dir=TRAIN_IMAGE_DIR,
                           metadata_path=TEST_METADATA_PATH,
                           retriever=retriever)

In [7]:
BATCH_SIZE = 16
NUM_WORKERS = 0

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

In [8]:
# import numpy as np
# from torch.utils.data import DataLoader, Subset

# num_train_samples = 1024
# num_test_samples = 128


# indices = np.random.choice(len(train_dataset), num_train_samples, replace=False)
# train_subset = Subset(train_dataset, indices)
# train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

# indices = np.random.choice(len(test_dataset), num_test_samples, replace=False)
# test_subset = Subset(test_dataset, indices)
# test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

In [10]:
model = create_default_FusionVLM().to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
# print(f"Total parameters: {num_params:,}\nText Decoder", end=' ')
# model.text_decoder.print_trainable_parameters()

c:\Users\Mahan\Documents\Projects\Retrieval-Augmented-Image-Captioning\.venv\Lib\site-packages\peft\tuners\tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


In [13]:
# model = FusionVLM(vision_encoder_name=CLIP_MODEL_NAME,
#                     text_encoder_name=CLIP_MODEL_NAME,
#                     T5_text_decoder_name=T5_MODEL_NAME,
#                     num_fusion_blocks=4,
#                     use_local_files=True
#                     )
# model = apply_lora_config(model).to(DEVICE)


In [11]:
print_model_param_stats(model)

Module                                          Total    Trainable       Frozen
--------------------------------------------------------------------------------
vision_encoder                             87,456,000            0   87,456,000
text_encoder                              109,628,544            0  109,628,544
vision_proj                                   590,592      590,592            0
text_proj                                     590,592      590,592            0
fusion_blocks                              56,724,480   56,724,480            0
post_fusion_ln                                  1,536        1,536            0
fusion_proj                                   590,592      590,592            0
text_decoder                              254,655,744   31,752,192  222,903,552
--------------------------------------------------------------------------------
TOTAL                                     510,238,080   90,249,984  419,988,096


In [12]:
train_dataset[0]

{'query_image': <PIL.Image.Image image mode=RGB size=333x500>,
 'retrieved_image': <PIL.Image.Image image mode=RGB size=375x500>,
 'context': 'Similar images are described as: \n A person in jeans and a t-shirt is enjoying spending time in a small garden outside . Two people are planting small bushes near a grassy area . Two men using gardening tools on the ground .',
 'target_caption': 'A picture of Two young guys with shaggy hair look at their hands while hanging out in the yard .',
 'all_captions': ['A picture of Two young guys with shaggy hair look at their hands while hanging out in the yard .',
  'A picture of Two young  White males are outside near many bushes .',
  'A picture of Two men in green shirts are standing in a yard .',
  'A picture of A man in a blue shirt standing in a garden .',
  'A picture of Two friends enjoy time spent together .']}

In [13]:
NUM_EPOCHS = 2
full_history = {}
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=0.01
)

setup_nltk()
os.makedirs(VLM_CHECKPOINT_DIR, exist_ok=True)

In [14]:
history = train_and_evaluate_model(model, train_loader, optimizer, NUM_EPOCHS, test_loader, T5_tokenizer)
add_dict(full_history, history)

Epoch 1:   0%|          | 0/1924 [00:00<?, ?it/s]

Epoch 1:   2%|▏         | 32/1924 [00:17<17:08,  1.84it/s, loss=5.52]


KeyboardInterrupt: 

In [ ]:
save_FusionVLM(model, f'epoch2', VLM_CHECKPOINT_DIR)

In [ ]:
full_history

In [ ]:
import json
with open('history.json', "w", encoding="utf-8") as f:
    json.dump(full_history, f, indent=2, ensure_ascii=False)

In [ ]:
model = load_default_FusionVLM('epoch2')
model = model.to(DEVICE)

In [19]:
with torch.no_grad():
    for batch in test_loader:
        gt_captions = batch["all_captions"]  # List[List[str]]

        generated_ids = model.generate(
            query_pixel_values=batch["query_pixel_values"],
            # retrieved_pixel_values=batch["retrieved_pixel_values"],
            retrieved_pixel_values=batch.get("retrieved_pixel_valuess"),
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            max_length=64,
            num_beams=1,
            # do_sample=True,
            # top_p=0.9,
            # temperature=0.8,
            # repetition_penalty=1.2,
        )

        decoded = T5_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        for i in range(len(decoded)):
            print(gt_captions[i])
            print(decoded[i])            
        break

['A picture of A female archer in a white uniform is aiming her arrow at a target far away .', 'A picture of A modern sport archer draws back an arrow  taking aim .', 'A picture of A female archer focusing before releasing a shot .', 'A picture of An oriental woman in glasses  shooting archery .', 'A picture of The girl is preparing to take her shot .']
A picture ofly a man holding a hat and a hat.
['A picture of A group of brown dogs are standing on a road with 3 people .', 'A picture of There are 5 brown dogs on leashes with their owners nearby .', 'A picture of Several brown dogs of different sizes gather together .', 'A picture of Three women are standing among a group of brown dogs .', 'A picture of Dogs and their masters gather on a dirt trail .']
A picture of a group of dogs is a picture of a group of dogs snooping around the neighborhood.
['A picture of Two dogs  one black  the other black and white runs on the beach .', 'A picture of A black dog and a black and white dog are r